### Model integration with OpenAI - Gemini - Groq

In [ ]:
import os
from dotenv import find_dotenv, load_dotenv

Loaded env from: c:\gitrepo\GitHub\langchain_v1\.env


In [ ]:
# Always reload latest values from .env into the current notebook kernel.
ENV_PATH = find_dotenv(filename=".env", usecwd=True)
if not ENV_PATH:
    raise FileNotFoundError("Could not locate a .env file from the current working directory tree.")

load_dotenv(dotenv_path=ENV_PATH, override=True)
print(f"Loaded env from: {ENV_PATH}")

In [ ]:
# OpenAI client setup

from pathlib import Path
from openai import OpenAI
import httpx

cert_env = os.environ.get("CERT_LLM", "").strip()
if not cert_env:
    raise EnvironmentError(
        "CERT_LLM is not set. Point it to your CA bundle file or directory."
    )

cert_path = Path(cert_env).expanduser()
# If CERT_LLM is a directory, use the default bundle filename inside it.
if cert_path.is_dir():
    cert_path = cert_path / "ca-bundle_lite.crt"

SSL_CERT = str(cert_path)
if not cert_path.exists():
    raise FileNotFoundError(f"Certificate bundle not found at: {SSL_CERT}")

BASE_URL = "https://litellm.icp.infineon.com"
MODEL = os.environ.get("OPENAI_MODEL", "gpt-5.2").strip()

http_client = httpx.Client(verify=SSL_CERT)
client = OpenAI(
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)

model_list = client.models.list()
available_models = [m.id for m in model_list.data if getattr(m, "id", None)]
valid_models = [m for m in available_models if m != "no-default-models"]

if MODEL == "no-default-models":
    MODEL = ""

if MODEL and MODEL not in available_models:
    print(f"Requested model '{MODEL}' is not available for this key.")
    MODEL = valid_models[0] if valid_models else ""

if not MODEL:
    print("No valid model available from the proxy.")
    print("Set OPENAI_MODEL to a model your key can access and rerun this cell.")
    print(f"Models returned by /v1/models: {available_models}")
else:
    print(f"Using model: {MODEL}")
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "this is a test request, write a short poem",
            }
        ],
    )
    print(response.choices[0].message.content)

Using model: gpt-5.2
A quiet thought across the page,  
A spark that hums inside a line—  
Small words arranged like lanterns raised  
To light a moment, briefly mine.


In [ ]:
from langchain.chat_models import init_chat_model

# Route through the LiteLLM proxy (reuses the SSL bundle + key); a bare model name
# would default to the public OpenAI endpoint, which the network refuses.
model = init_chat_model(
    MODEL,
    model_provider="openai",
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9', 'langchain-openai': '1.3.2'}}, output_version=None, profile={'name': 'GPT-5.2', 'release_date': '2025-12-11', 'last_updated': '2025-12-11', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x0000021D6C9B69E0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000021D6C6EF890>, root_client=<openai.OpenAI object at 0

In [ ]:
# invoke model
response = model.invoke("Hello, how are you?")
response

AIMessage(content='I’m here and ready to help. What would you like to work on today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 12, 'total_tokens': 33, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 9, 'engine_ttft_ms': 53, 'engine_ttlt_ms': 248, 'pre_inference_ms': 103, 'service_tbt_ms': 9, 'service_ttft_ms': 315, 'service_ttlt_ms': 506, 'total_duration_ms': 412, 'user_visible_ttft_ms': 213}}, 'model_provider': 'openai', 'model_name': 'gpt-5.2', 'system_fingerprint': None, 'id': 'chatcmpl-DtVTR5eWX2NJfvtccyuhBHSLKhgKW', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eeeb6-7abf-7c51-adea-09838b31c6d1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 12, 'out

In [ ]:
response.content

'I’m here and ready to help. What would you like to work on today?'

### ChatOpenAI

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model=MODEL,
    base_url=f"{BASE_URL}/v1",
    api_key=os.environ.get("OPENAI_API_KEY"),
    http_client=http_client,
)

response = model.invoke("Hello, how are you?")
response.content

'I’m here and ready to help. What would you like to work on today?'

### Gemini Model integration

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")
response = model.invoke("Hello, how are you?")
response.content

"Hello! I'm doing well, thank you for asking. I'm a large language model, so I don't experience feelings in the same way humans do, but I'm functioning optimally and ready to assist you.\n\nHow are you today? What can I help you with?"

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
response = model.invoke("Hello, how are you?")
response.content

'Hello! I am functioning well, thank you for asking. I am a large language model, trained by Google. How can I help you today?'

### Groq Model integration

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Hello, how are you?")
response.content

'<think>\nOkay, let\'s see. The user said "Hello, how are you?" I need to respond appropriately. First, I should acknowledge their greeting and offer a friendly response. Since I\'m an AI, I don\'t have feelings, but I should express that in a way that\'s engaging and not too robotic. Maybe start with a cheerful "Hi!" to match their tone. Then, mention that I\'m an AI without feelings but ready to help. Keep it simple and conversational. Also, ask how I can assist them to keep the conversation going. Let me make sure the response is warm and inviting.\n</think>\n\nHi! 😊 I\'m an AI, so I don\'t have feelings in the traditional sense, but I\'m here and ready to help! How can I assist you today?'

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="qwen/qwen3-32b")
response = model.invoke("Hello, how are you?")  
response.content

'<think>\nOkay, the user is asking "Hello, how are you?" I need to respond in a friendly and positive way. First, I should acknowledge their greeting and offer my own. Then, express that I\'m here to help and ask how I can assist them. Keep it simple and conversational, no need for complex language. Make sure to be polite and open-ended to invite them to ask for help with anything specific. Also, check if there\'s any additional context I should consider, but since it\'s a straightforward greeting, probably not. Alright, let\'s put that together.\n</think>\n\nHello! I\'m doing well, thank you! 😊 How can I assist you today?'